# Transformer Encoder and Dense Retrieval — Change Proposal & Validation Notebook

**Project:** 02456 Deep Learning Project — Dense Passage Retrieval with Transformer Encoders  
**Review target:** the supplied `transformer-encoders` repository snapshot (16 November 2026), compared with the supplied starter repository. This notebook is a change proposal and validation harness; it does not modify the repository. The proposed code is a review/reference patch, not an automatic modification of the repository.  
**Purpose:** give the team a technically grounded change proposal, reproduce concrete failures, show replacement implementations, and specify exactly what should change in the repository.

> **How to run this notebook:** Put this notebook in the **same directory as `main.py`**. Start Jupyter with that directory as its working directory (or set `PROJECT_ROOT` manually in the first code cell). The notebook adds the project root to `sys.path`, so imports such as `from src...` work in the same way as in `main.py`.

This notebook does not silently edit the repository. Proposed code is shown in cells for review and testing. Once agreed, the changes can be copied into the indicated files.

## Review principles

- **A — Current implementation:** exact or faithful excerpts from the supplied fork.
- **B — Why it is wrong / risky:** the failure mode and its likely impact.
- **C — Proposed implementation:** replacement code or a concrete patch.
- **D — Why the replacement is correct:** the relevant invariant or expected behavior.
- **E — Repository action:** the file(s) and exact change required.

A finding marked **confirmed** follows directly from supplied code and/or a reproduced test. A finding marked **design choice** is not necessarily a bug; it is a recommendation or experiment. Dataset statements are separated from code inspection and linked to the relevant dataset cards.

## 1. Executive summary

The core Transformer encoder in the fork is substantially implemented. The multi-head tensor reshaping, scaled dot-product attention, feed-forward network, post-norm residual blocks, and mask-aware mean/max pooling are structurally reasonable. The existing tests pass, but this only validates the tested component behavior.

The principal blockers are **outside the core attention equations**:

1. The project-level `DPR` class cannot construct its encoders correctly and does not retain them as model attributes.
2. `DPRLoss` and `Trainer` are placeholders; there is no complete project training path.
3. The data pipeline conflates reference answers, passages and relevance judgements. In addition, the current `DocumentDataset` splits ordinary string-valued rows into characters.
4. The tokenizer/vocabulary contract is not connected to DPR or the encoder configuration.
5. The implemented TF-IDF/BM25 baselines contain reproducible errors.
6. Recall is calculated with an incorrect denominator in some cases, and the requested Precision@k and MRR are not implemented in the evaluator.
7. `main.py` always constructs TF-IDF, regardless of the configured retriever.

**Conclusion:** do not rewrite the entire Transformer on the assumption that attention is fundamentally broken. First harden the encoder edge cases, then fix data contracts, DPR, training, and evaluation. Only after those changes should model quality be assessed on a real retrieval benchmark.

| Area | Current status | Priority |
|---|---|---|
| Encoder core equations | Structurally sound; one public API edge case | Medium |
| Attention-map access | Calculated internally but discarded by encoder blocks | Medium / assignment requirement |
| DPR model construction | Placeholder / cannot run | Critical |
| DPR loss and trainer | Not implemented | Critical |
| FiQA data/labels | Current loader uses answers as pseudo-documents; lacks qrels | Critical |
| SQuAD processing | String context iterated as characters | Critical |
| DocumentDataset | String rows are zipped character-by-character | Critical |
| Tokenization/config | Not integrated with DPR | Critical |
| TF-IDF/BM25 | Several reproducible implementation errors | High |
| Evaluation | Recall denominator issue; Precision@k and MRR absent | High |
| Main dispatch | TF-IDF is hard-coded | High |

This review concerns the supplied 16-Nov-2026 fork snapshot. Re-run the validation cells against the exact branch/commit used by the team before merging changes.

## 2. Environment and project-root setup

The next cell expects `main.py` and `src/` to be in the current working directory. If the notebook is launched elsewhere, set `PROJECT_ROOT` to the absolute repository path.

In [3]:
from pathlib import Path
import os
import sys

# Default assumes the notebook was launched from the directory containing main.py.
PROJECT_ROOT = Path.cwd()

# You may override this explicitly, for example:
# PROJECT_ROOT = Path("/absolute/path/to/02456-retrieval-project")

if not (PROJECT_ROOT / "main.py").exists() or not (PROJECT_ROOT / "src").is_dir():
    candidates = [p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "main.py").exists() and (p / "src").is_dir()]
    if candidates:
        PROJECT_ROOT = candidates[0]
    else:
        raise RuntimeError(
            "Could not find main.py and src/. Put this notebook beside main.py, "
            "launch Jupyter from that directory, or set PROJECT_ROOT manually."
        )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("main.py exists:", (PROJECT_ROOT / "main.py").exists())
print("src/ exists:", (PROJECT_ROOT / "src").is_dir())

Project root: /home/husted42/Documents/github/transformer-encoders
main.py exists: True
src/ exists: True


In [4]:
import importlib.util
import platform
import torch
import numpy as np

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
for package in ["pytest", "scipy", "datasets", "transformers", "hydra"]:
    print(f"{package:12s}", "available" if importlib.util.find_spec(package) else "not installed")

ModuleNotFoundError: No module named 'torch'

### 2.1 Re-run the fork's supplied tests

The fork contains one encoder test and seven vector-database tests. These validate shapes, padding invariance, gradients and index operations. They do **not** test actual FiQA/SQuAD DPR training.

In [ ]:
import pytest

existing_test_paths = [
    PROJECT_ROOT / "tests" / "test_encoder.py",
    PROJECT_ROOT / "tests" / "test_vector_database.py",
]
missing = [str(p) for p in existing_test_paths if not p.exists()]
if missing:
    print("The supplied test files were not found:", missing)
else:
    result = pytest.main(["-q", *map(str, existing_test_paths)])
    print("pytest exit code:", result)
    assert result == 0, "The existing fork test suite did not pass."

.

.

.

.

.

.

.

.

                                                                 [100%]


8 passed in 0.09s


pytest exit code: 0


**Observed baseline in the supplied snapshot:** `8 passed`. If this notebook is run against a different commit and the result differs, record the commit hash and investigate before treating the rest of this proposal as validated.

## 3. Transformer encoder: hardening, not a wholesale rewrite

### 3.1 Fully masked attention rows can produce NaNs

#### A — Current implementation

`src/retrievers/models/encoder.py`, `ScaledDotProductAttention.forward`:

```python
scores = q @ k.transpose(-2,-1)
scaled_scores = scores/sqrt(q.shape[-1])
if mask is not None:
    scaled_scores = scaled_scores.masked_fill(~mask,float("-inf"))
attention_weights = F.softmax(scaled_scores,dim=-1)
output = attention_weights @ v
return (output,attention_weights)
```

#### B — Why it is risky

If every key is masked for a particular query row, that row becomes `[-inf, -inf, ...]`. Softmax of that row is undefined and returns NaNs. The full `TransformerEncoder.forward` currently rejects a completely padded sequence, so normal right-padded batches avoid this case. However, `ScaledDotProductAttention` is a public component and its documented mask contract does not itself prevent an all-false row.

This is a robustness defect, not evidence that ordinary multi-layer encoder passes accumulate numerical errors. The replacement below fails fast with an informative error rather than silently propagating NaNs.

#### C — Proposed implementation

Replace the attention calculation with the following validated function. It supports masks that broadcast to the attention-score shape and explicitly rejects rows with no allowed key.

In [ ]:
import math
import torch
import torch.nn.functional as F

def safe_scaled_dot_product_attention(q, k, v, mask=None):
    """Scaled dot-product attention; mask=True means a key is allowed."""
    if q.ndim < 2 or k.ndim != q.ndim or v.ndim != q.ndim:
        raise ValueError("q, k and v must have the same rank and at least 2 dimensions")
    if q.shape[:-2] != k.shape[:-2] or k.shape[:-2] != v.shape[:-2]:
        raise ValueError("q, k and v leading dimensions must match")
    if q.shape[-1] != k.shape[-1]:
        raise ValueError("q and k feature dimensions must match")
    if k.shape[-2] != v.shape[-2]:
        raise ValueError("k and v sequence lengths must match")
    if not (q.is_floating_point() and k.is_floating_point() and v.is_floating_point()):
        raise TypeError("q, k and v must be floating-point tensors")
    if q.device != k.device or k.device != v.device:
        raise ValueError("q, k and v must be on the same device")

    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(q.shape[-1])

    if mask is not None:
        if mask.dtype != torch.bool:
            raise TypeError("mask must be boolean: True=allowed, False=blocked")
        try:
            allowed = torch.broadcast_to(mask.to(device=scores.device), scores.shape)
        except RuntimeError as exc:
            raise ValueError(
                f"mask shape {tuple(mask.shape)} is not broadcastable to "
                f"attention shape {tuple(scores.shape)}"
            ) from exc
        if not allowed.any(dim=-1).all():
            raise ValueError("Every attention row must allow at least one key")
        scores = scores.masked_fill(~allowed, torch.finfo(scores.dtype).min)

    weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(weights, v)
    return output, weights

#### D — Why the replacement is correct

- The scale remains \(1/\sqrt{d_k}\), where \(d_k\) is the query/key head width.
- `True` continues to mean “allowed”, matching the current encoder mask convention.
- Mask broadcasting is checked before use.
- Every softmax row must contain at least one valid key, so softmax cannot receive an all-masked row.
- Valid masked positions receive zero attention probability.

#### E — Repository change

Edit `src/retrievers/models/encoder.py`, replacing `ScaledDotProductAttention.forward` with equivalent logic. Keep the public return order `(output, attention_weights)` so existing callers remain compatible. Add the tests below to `tests/test_encoder.py`.

In [ ]:
# Validation: unmasked attention sums to one, padding gets zero mass,
# and a fully masked row is rejected instead of generating NaNs.
torch.manual_seed(7)
q = torch.randn(2, 3, 8)
k = torch.randn(2, 4, 8)
v = torch.randn(2, 4, 6)
mask = torch.tensor([[[True, True, False, False]], [[True, False, True, False]]])
out, weights = safe_scaled_dot_product_attention(q, k, v, mask)
assert out.shape == (2, 3, 6)
assert weights.shape == (2, 3, 4)
assert torch.isfinite(out).all() and torch.isfinite(weights).all()
assert torch.allclose(weights.sum(-1), torch.ones(2, 3), atol=1e-6)
assert torch.all(weights.masked_select(~mask.expand_as(weights)) == 0)

try:
    safe_scaled_dot_product_attention(q[:1], k[:1], v[:1], torch.zeros(1, 1, 4, dtype=torch.bool))
except ValueError as exc:
    print("fully masked row rejected as expected:", exc)
else:
    raise AssertionError("An all-masked row should have raised ValueError")
print("safe attention tests: PASS")

fully masked row rejected as expected: Every attention row must allow at least one key
safe attention tests: PASS


### 3.2 Attention weights are discarded by the encoder stack

#### A — Current implementation

`MultiHeadSelfAttention.forward` returns `(output, attention_weights)`, but `EncoderBlock.forward` currently does:

```python
attention_output, _ = self.attention(x, mask)
x = self.norm1(x + self.dropout1(attention_output))
...
return x
```

`TransformerEncoder.forward` loops over the blocks and only returns token or pooled embeddings.

#### B — Why it matters

This is not a retrieval correctness bug. It is an observability gap: attention weights are computed but discarded before the model output. The assignment specifically asks for qualitative attention-map examination, so the complete encoder needs an opt-in path to return maps from every layer and head.

The current encoder also documents that contextual outputs at padded token positions are not zeroed. Pooling correctly masks them, but returning token embeddings creates a risk that a downstream caller will accidentally pool or visualize padded outputs as if they were real tokens.

#### C — Proposed implementation

Change the encoder block to optionally return its attention weights and zero the padded token states after each block. Keep the default return type unchanged for backward compatibility.

In [ ]:
# Replacement pattern for EncoderBlock.forward (insert into EncoderBlock).
def encoder_block_forward_with_attentions(self, x, mask=None, return_attention=False):
    attention_output, attention_weights = self.attention(x, mask)
    x = self.norm1(x + self.dropout1(attention_output))
    ffn_output = self.feed_forward(x)
    x = self.norm2(x + self.dropout2(ffn_output))

    # mask has shape (B, 1, L); make it (B, L, 1) for token states.
    if mask is not None:
        x = x.masked_fill(~mask.transpose(1, 2), 0.0)

    if return_attention:
        return x, attention_weights
    return x

# Replacement pattern for the TransformerEncoder layer loop / return section.
def collect_encoder_layers(encoder, x, mask, return_token_embeddings=False, output_attentions=False):
    all_attention_weights = []
    for layer in encoder.layers:
        if output_attentions:
            x, layer_weights = encoder_block_forward_with_attentions(
                layer, x, mask, return_attention=True
            )
            all_attention_weights.append(layer_weights)
        else:
            x = encoder_block_forward_with_attentions(layer, x, mask)

    output = x if return_token_embeddings else encoder.pool(x, mask)
    if output_attentions:
        return output, tuple(all_attention_weights)
    return output

#### D — Why the replacement is correct

- The default path still returns the same tensor as before.
- The optional attention path returns one `(batch, heads, sequence, sequence)` tensor per layer.
- Zeroing padded query states does not change valid-token outputs: valid query positions already cannot attend to padded keys, and the feed-forward network acts independently at each position.
- The pooling code continues to use the explicit validity mask.

The two functions above are patch patterns, not a drop-in replacement for the whole class: move the `encoder_block_forward_with_attentions` logic into `EncoderBlock.forward`, then add an `output_attentions=False` parameter to `TransformerEncoder.forward` and use the loop pattern there.

#### E — Repository change

Edit `EncoderBlock.forward` and `TransformerEncoder.forward` in `src/retrievers/models/encoder.py`. Add an optional `output_attentions` argument. Add tests that verify one attention tensor per layer, finite values, zero probability on padded keys, and unchanged default output shape.

### 3.3 Validate the complete encoder stack for finite outputs and gradients

The existing tests cover gradient propagation on a small stack. This extra stress test uses more layers and checks both forward and backward passes. It is a numerical smoke test, not a quality benchmark.

In [ ]:
from src.retrievers.models.encoder import TransformerEncoder

torch.manual_seed(11)
encoder = TransformerEncoder(
    vocab_size=100, d_model=64, n_heads=4, n_layers=8,
    feed_forward_dim=256, max_len=64, dropout=0.1, pooling_type="mean"
)
tokens = torch.tensor([
    [2, 5, 9, 12, 0, 0],
    [2, 7, 8, 3, 4, 0],
    [2, 11, 0, 0, 0, 0],
], dtype=torch.long)
encoder.train()
embeddings = encoder(tokens)
loss = embeddings.square().mean()
loss.backward()

assert embeddings.shape == (3, 64)
assert torch.isfinite(embeddings).all()
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in encoder.parameters())
print("8-layer forward/backward finite-value test: PASS")
print("embedding norms:", embeddings.norm(dim=1).detach().cpu().numpy().round(3))

8-layer forward/backward finite-value test: PASS
embedding norms: [6.62 6.52 7.33]


### 3.4 Encoder architecture choices that are not confirmed bugs

The current encoder adds token embeddings and sinusoidal positions without multiplying token embeddings by \(\sqrt{d_{model}}\), and it does not apply dropout immediately after that sum. The original Transformer used embedding scaling and dropout, but their absence is not proof of a broken implementation. Treat both as **controlled ablations**:

- Baseline: current embedding + positional encoding.
- Variant A: multiply token embeddings by `sqrt(d_model)` before adding positions.
- Variant B: add dropout after the embedding/position sum.

Compare validation retrieval metrics with the same data, tokenizer, optimizer, seed and training budget. Do not change both at once. The project can document the selected design rather than present an architectural preference as a bug.

## 4. Encoder configuration and pooling contract

### 4.1 The YAML does not describe the new encoder constructor

#### A — Current implementation

`config/model/encoder.yaml`:

```yaml
pooling_type: 'cls'
emb_dim: 128
hidden_dim: 32
n_heads: 4
n_layers: 6
feed_forward_dim: 768
```

But `TransformerEncoder.__init__` expects `vocab_size`, `d_model`, `n_heads`, `n_layers`, `feed_forward_dim`, and `max_len`. The experiment config also does not include the `model` config group in its Hydra defaults.

#### B — Why it is wrong

The config is stale relative to the new class:

- `emb_dim` is named `d_model` in the encoder.
- `hidden_dim` is not consumed by the new encoder.
- `vocab_size` and `max_len` are missing.
- `pooling_type: cls` only works if the tokenizer adds a real CLS token at position zero.
- The default experiment config does not compose `model/encoder.yaml`, and `main.py` does not call DPR anyway.

Thus the YAML currently does not configure a working DPR model.

#### C — Proposed implementation

For a scratch-trained encoder, make the dimensions explicit and bind `vocab_size` to the tokenizer vocabulary at model construction time. Do not hard-code a guessed vocabulary size in YAML.

```yaml
# config/model/encoder.yaml; vocab_size is supplied by the tokenizer at runtime.
d_model: 128
n_heads: 4
n_layers: 6
feed_forward_dim: 512
max_len: 256
padding_idx: 0
pooling_type: mean       # Recommended first baseline for a scratch encoder.
dropout: 0.1
```

#### D — Why it is correct

The fields now match the constructor and the vocabulary size remains owned by the tokenizer. Starting with masked mean pooling avoids silently relying on a CLS convention that the current tokenization pipeline does not implement. CLS pooling is valid too, but only when `[CLS]` is prepended consistently for both queries and passages.

#### E — Repository change

1. Rename `emb_dim` to `d_model` in `config/model/encoder.yaml`.
2. Remove `hidden_dim` unless a new layer actually uses it.
3. Add `max_len`, `padding_idx`, and `dropout`.
4. Add `- model: encoder` to the Hydra defaults in `config/fiqa_experiment.yaml`.
5. Build the encoder with `vocab_size=tokenizer.vocab_size` in the DPR model constructor.
6. Keep `pooling_type: cls` only if the tokenizer always inserts a CLS token and the tests verify it.

## 5. DPR model, contrastive loss and trainer

### 5.1 DPR model construction and checkpoint loading

#### A — Current implementation

`src/retrievers/dpr.py`:

```python
query_encoder = TransformerEncoder(...)
document_encoder = TransformerEncoder(...)

query_weights_path = self.cfg.model.get('query_encoder_weights_path', None)
if self.cfg.model.get('query_encoder_weights_path', None):
    query_encoder.load_state_dict(query_weights_path)

if self.cfg.model.get('query_encoder_weights_path', None):
    document_encoder.load_state_dict(document_weights_path)
```

#### B — Why it is wrong

- `TransformerEncoder(...)` contains `...` and cannot be instantiated.
- The constructed encoders are local variables, not `self.query_encoder` / `self.document_encoder` attributes.
- `load_state_dict` expects a state dictionary, not a filesystem path.
- The document checkpoint condition incorrectly checks the query checkpoint setting.
- The class does not inherit from `nn.Module`, so it cannot be moved, optimized, or checkpointed as a single model.

#### C — Proposed implementation

Use an `nn.Module` containing two independent encoder instances. The same config can initialize both, while checkpoints are loaded independently.

In [ ]:
# Reproduce the supplied DPR constructor failure without editing repository files.
from types import SimpleNamespace
from src.retrievers.dpr import DPR as CurrentDPR
from src.training.loss import DPRLoss as CurrentDPRLoss
from src.training.trainer import Trainer as CurrentTrainer

try:
    CurrentDPR(SimpleNamespace(model={}))
except TypeError as exc:
    print("current DPR construction fails as expected:", str(exc).splitlines()[0])
else:
    raise AssertionError("Expected the supplied DPR constructor to fail")

try:
    CurrentDPRLoss()(torch.randn(2, 8), torch.randn(2, 8))
except NotImplementedError:
    print("current DPRLoss NotImplementedError reproduced: PASS")
else:
    raise AssertionError("Expected supplied DPRLoss to be unimplemented")

try:
    CurrentTrainer().train(torch.nn.Linear(1, 1), torch.nn.Linear(1, 1))
except TypeError as exc:
    print("current Trainer method signature failure reproduced:", str(exc).splitlines()[0])
else:
    raise AssertionError("Expected the supplied Trainer.train signature to fail")


current DPR construction fails as expected: TransformerEncoder.__init__() missing 5 required positional arguments: 'd_model', 'n_heads', 'n_layers', 'feed_forward_dim', and 'max_len'
current DPRLoss NotImplementedError reproduced: PASS
current Trainer method signature failure reproduced: Trainer.train() takes 2 positional arguments but 3 were given


In [ ]:
import torch
import torch.nn as nn
from pathlib import Path
from src.retrievers.models.encoder import TransformerEncoder

class DPRModel(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        encoder_config: dict,
        query_checkpoint: str | None = None,
        document_checkpoint: str | None = None,
    ):
        super().__init__()
        self.encoder_config = dict(encoder_config)
        self.query_encoder = TransformerEncoder(vocab_size=vocab_size, **self.encoder_config)
        self.document_encoder = TransformerEncoder(vocab_size=vocab_size, **self.encoder_config)

        self._load_encoder_checkpoint(self.query_encoder, query_checkpoint)
        self._load_encoder_checkpoint(self.document_encoder, document_checkpoint)

    @staticmethod
    def _load_encoder_checkpoint(encoder: nn.Module, checkpoint_path: str | None) -> None:
        if checkpoint_path is None:
            return
        path = Path(checkpoint_path)
        if not path.is_file():
            raise FileNotFoundError(path)
        state = torch.load(path, map_location="cpu")
        # Support either a bare state_dict or a common checkpoint wrapper.
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        encoder.load_state_dict(state)

    def forward(self, query_token_ids: torch.Tensor, document_token_ids: torch.Tensor):
        query_vectors = self.query_encoder(query_token_ids)
        document_vectors = self.document_encoder(document_token_ids)
        return query_vectors, document_vectors

#### D — Why it is correct

The two towers are registered model submodules, so their parameters appear in `model.parameters()`, gradients flow to both, and `state_dict()` contains both towers. Checkpoint paths are loaded to actual dictionaries, each tower gets its own checkpoint condition, and model/device management works through ordinary PyTorch APIs.

#### E — Repository change

Replace `src/retrievers/dpr.py` with an `nn.Module` implementation similar to the code above. Add explicit model config composition to Hydra. Do not load a document checkpoint based on the query checkpoint flag. Add a test that checks both towers exist, both produce `(batch, d_model)` vectors, and both have trainable parameters.

### 5.2 DPR contrastive loss is missing

#### A — Current implementation

`src/training/loss.py`:

```python
class DPRLoss(nn.Module):
    def forward(self, queries, documents):
        raise NotImplementedError()
```

#### B — Why it is wrong

There is no project-level objective that makes matched query/passage pairs score higher than other passages. The synthetic demo independently implements `cross_entropy(query_vectors @ document_vectors.T, diagonal_labels)`, but that does not implement the repository's `DPRLoss` class and is not wired to the repository trainer.

A diagonal in-batch loss also assumes that document column `i` is positive for query row `i`, and all other columns are negatives. This can create **false negatives** when a batch contains another relevant passage for a query. The code below supports both the one-positive diagonal case and a multi-positive mask.

#### C — Proposed implementation

The assignment defines an inner-product scoring objective. Use dot products, divide by a configurable temperature, and cross-entropy for one positive per query. If a batch has multiple known positive documents per query, pass a boolean `positive_mask`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DPRLoss(nn.Module):
    def __init__(self, temperature: float = 1.0):
        super().__init__()
        if temperature <= 0:
            raise ValueError("temperature must be positive")
        self.temperature = float(temperature)

    def forward(self, query_vectors, document_vectors, positive_mask=None):
        if query_vectors.ndim != 2 or document_vectors.ndim != 2:
            raise ValueError("query and document vectors must be rank-2")
        if query_vectors.shape[1] != document_vectors.shape[1]:
            raise ValueError("query and document embedding dimensions must match")
        if query_vectors.shape[0] == 0 or document_vectors.shape[0] == 0:
            raise ValueError("query and document batches must be nonempty")

        logits = (query_vectors @ document_vectors.T) / self.temperature

        if positive_mask is None:
            if query_vectors.shape[0] != document_vectors.shape[0]:
                raise ValueError("diagonal positives require equal query/document batch sizes")
            labels = torch.arange(query_vectors.shape[0], device=logits.device)
            return F.cross_entropy(logits, labels)

        if positive_mask.dtype != torch.bool or positive_mask.shape != logits.shape:
            raise ValueError("positive_mask must be boolean with shape (n_queries, n_documents)")
        positive_mask = positive_mask.to(logits.device)
        if not positive_mask.any(dim=1).all():
            raise ValueError("Every query must have at least one positive document in the batch")

        log_denominator = torch.logsumexp(logits, dim=1)
        log_numerator = torch.logsumexp(logits.masked_fill(~positive_mask, float("-inf")), dim=1)
        return -(log_numerator - log_denominator).mean()

#### D — Why it is correct

For one positive passage at each diagonal position, the loss is exactly the batch cross-entropy used in the synthetic demo. For multi-positive batches, the numerator sums the probability mass over all known positive columns, so a known relevant passage is not incorrectly treated as a negative. The loss remains differentiable and the implementation follows the inner-product objective described in the assignment.

Temperature is an explicit training hyperparameter; `1.0` preserves the unscaled inner-product objective. Tune it only on validation data.

#### E — Repository change

Replace `src/training/loss.py` with the implementation above. Add tests for: diagonal positives, multi-positive masks, invalid shapes, missing positives, and nonzero finite gradients.

In [ ]:
# Validate DPRLoss: identity-matched pairs should be the easiest diagonal positives.
loss_fn = DPRLoss(temperature=1.0)
q_vecs = torch.eye(4, requires_grad=True)
d_vecs = torch.eye(4, requires_grad=True)
loss = loss_fn(q_vecs, d_vecs)
loss.backward()
assert torch.isfinite(loss)
assert q_vecs.grad is not None and torch.isfinite(q_vecs.grad).all()

# Query 0 has two positive documents; query 1 has one positive document.
positive_mask = torch.tensor([[True, False, True, False],
                              [False, True, False, False]])
multi_loss = loss_fn(q_vecs.detach()[:2], d_vecs.detach(), positive_mask=positive_mask)
assert torch.isfinite(multi_loss)
print("single-positive loss:", float(loss.detach()))
print("multi-positive loss:", float(multi_loss.detach()))
print("DPR loss / gradient checks: PASS")

single-positive loss: 0.7436683773994446
multi-positive loss: 0.5870375037193298
DPR loss / gradient checks: PASS


### 5.3 The trainer is a placeholder, not a training system

#### A — Current implementation

`src/training/trainer.py`:

```python
class Trainer:
    def __init__(self):
        self.criterion = DPRLoss()

    def train(query_encoder, document_encoder):
        raise NotImplementedError()
```

#### B — Why it is wrong

`train` has no `self` parameter, no data loader, no optimizer, no loss call, no `zero_grad`, no `backward`, no optimizer step, no validation, and no checkpoint saving. The repository therefore has no functioning project trainer.

#### C — Proposed implementation

The following is a small, auditable trainer for batches containing `query_input_ids`, `document_input_ids`, and optionally `positive_mask`. It assumes the model implements the `DPRModel.forward` contract above.

In [ ]:
class DPRTrainer:
    def __init__(self, model, optimizer, criterion=None, device="cpu", grad_clip_norm=1.0):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.criterion = criterion or DPRLoss()
        self.device = torch.device(device)
        self.grad_clip_norm = grad_clip_norm

    def train_epoch(self, batches):
        self.model.train()
        total_loss, n_batches = 0.0, 0
        for batch in batches:
            q_ids = batch["query_input_ids"].to(self.device)
            d_ids = batch["document_input_ids"].to(self.device)
            positive_mask = batch.get("positive_mask")
            if positive_mask is not None:
                positive_mask = positive_mask.to(self.device)

            self.optimizer.zero_grad(set_to_none=True)
            q_vec, d_vec = self.model(q_ids, d_ids)
            loss = self.criterion(q_vec, d_vec, positive_mask=positive_mask)
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite DPR loss encountered")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip_norm)
            self.optimizer.step()
            total_loss += float(loss.detach())
            n_batches += 1
        if n_batches == 0:
            raise ValueError("Training loader produced no batches")
        return total_loss / n_batches

    @torch.no_grad()
    def evaluate_loss(self, batches):
        was_training = self.model.training
        self.model.eval()
        total_loss, n_batches = 0.0, 0
        try:
            for batch in batches:
                q_ids = batch["query_input_ids"].to(self.device)
                d_ids = batch["document_input_ids"].to(self.device)
                positive_mask = batch.get("positive_mask")
                if positive_mask is not None:
                    positive_mask = positive_mask.to(self.device)
                q_vec, d_vec = self.model(q_ids, d_ids)
                loss = self.criterion(q_vec, d_vec, positive_mask=positive_mask)
                total_loss += float(loss)
                n_batches += 1
        finally:
            self.model.train(was_training)
        if n_batches == 0:
            raise ValueError("Validation loader produced no batches")
        return total_loss / n_batches

#### D — Why the replacement is correct

This trainer explicitly performs the standard optimization sequence, checks for non-finite loss, clips gradients, supports validation without gradients, and restores the model's prior train/eval state after validation. The batch-level positive mask preserves support for multiple relevant documents.

This is a reference implementation, not a complete production trainer. The repository still needs a `Dataset`/`DataLoader`, sampler, scheduler, validation ranking evaluation, early stopping/checkpoint policy and experiment logging.

#### E — Repository change

Replace `src/training/trainer.py` with a trainer that takes `model`, `optimizer`, `criterion`, and `device` in its constructor. Implement the batch format above. Add a training smoke test that checks the loss is finite and at least one parameter changes after an optimizer step.

### 5.4 Test an end-to-end synthetic DPR training step

This test is deliberately small and artificial. It checks that the *proposed* model/loss/trainer interfaces connect and can take optimizer steps. It does not predict FiQA/SQuAD retrieval quality.

In [ ]:
# A tiny smoke test: independent towers, a contrastive objective, and optimizer steps.
torch.manual_seed(17)
config = dict(d_model=16, n_heads=4, n_layers=1, feed_forward_dim=32,
              max_len=8, padding_idx=0, dropout=0.0, pooling_type="mean")
model = DPRModel(vocab_size=40, encoder_config=config)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
trainer = DPRTrainer(model, optimizer, criterion=DPRLoss(temperature=1.0), device="cpu")

# Each row is a positive query-passage pair; other passages in the batch are negatives.
batches = []
for _ in range(3):
    q = torch.randint(2, 40, (4, 6))
    d = torch.randint(2, 40, (4, 6))
    q[:, 0] = 1  # reserved CLS-like marker in this smoke-test vocabulary
    d[:, 0] = 1
    batches.append({"query_input_ids": q, "document_input_ids": d})

before = [p.detach().clone() for p in model.parameters()]
train_loss = trainer.train_epoch(batches)
changed = any(not torch.equal(old, new.detach()) for old, new in zip(before, model.parameters()))
assert np.isfinite(train_loss) and changed
print(f"synthetic DPR training smoke test: PASS (mean batch loss={train_loss:.4f}; parameters updated={changed})")

synthetic DPR training smoke test: PASS (mean batch loss=1.4233; parameters updated=True)


## 6. Data pipeline: keep corpus, queries and relevance judgements separate

### 6.1 The current `Corpus` class mixes different data concepts

#### A — Current implementation

In `src/data_utils/corpus.py`, the current loader maps:

```python
column_mapping = {'fiqa': 'ground_truths', 'squad': 'context'}
...
documents = self.ds[key]['text'][i]
for doc in documents:
    relevant_ids.append(f'doc{doc_id_counter}')
```

`all_documents` then concatenates the generated `text` and `document_id` fields across train/validation/test splits.

#### B — Why it is wrong

**FiQA:** the fork loads `vibrantlabsai/fiqa` with configuration `main`. Its `ground_truths` field is a list of reference answers. Those answers are not the FiQA retrieval corpus and are not qrels. The same Hub dataset has a separate `corpus` configuration, but the repository does not load it. The Hub dataset card states that `main` contains questions and ground-truth answers, while `corpus` contains 57,638 documents. The `main` configuration does not provide query-to-document qrels. See the source links at the end of the notebook.

**SQuAD:** the repository maps `context` to `text`, but SQuAD's `context` is a string. Iterating over it yields characters. SQuAD is an extractive question-answering dataset: questions are paired with paragraphs and answer spans. For a retrieval adaptation, paragraphs can be candidate passages and each question can be linked to its source paragraph, but this is a constructed qrel, not an original open-corpus retrieval qrel.

**General design flaw:** query records, corpus passage records and qrels should be separate objects. They should not be inferred by iterating through the contents of a `text` field.

#### C — Proposed implementation

Use a normalized retrieval data contract:

- `corpus`: one row per passage with a stable `document_id` and a string `text`.
- `queries`: one row per query with a stable `query_id` and a string `text`.
- `qrels`: mapping `query_id -> {document_id: relevance_score}`.

For FiQA, use the BEIR FiQA benchmark data (corpus, queries and qrels) or another documented retrieval dataset with explicit judgements. For SQuAD, deduplicate paragraphs and create a documented question-to-source-paragraph relevance mapping.

In [ ]:
# Reference schema validator. This code runs without downloading any dataset.
def validate_retrieval_records(corpus_records, query_records, qrels):
    corpus_ids = [row["document_id"] for row in corpus_records]
    query_ids = [row["query_id"] for row in query_records]
    assert len(corpus_ids) == len(set(corpus_ids)), "document_id values must be unique"
    assert len(query_ids) == len(set(query_ids)), "query_id values must be unique"
    corpus_id_set, query_id_set = set(corpus_ids), set(query_ids)

    for row in corpus_records:
        assert isinstance(row["text"], str) and row["text"].strip(), "each passage needs nonempty string text"
    for row in query_records:
        assert isinstance(row["text"], str) and row["text"].strip(), "each query needs nonempty string text"
    for query_id, relevant_docs in qrels.items():
        assert query_id in query_id_set, f"qrels refer to unknown query: {query_id}"
        assert relevant_docs, f"query {query_id} has no positive judgement"
        for document_id, score in relevant_docs.items():
            assert document_id in corpus_id_set, f"qrels refer to unknown document: {document_id}"
            assert score > 0, "only positive relevance judgements belong in this positive-pair mapping"
    return True

fixture_corpus = [
    {"document_id": "d1", "text": "first passage"},
    {"document_id": "d2", "text": "second passage"},
]
fixture_queries = [
    {"query_id": "q1", "text": "first question"},
    {"query_id": "q2", "text": "second question"},
]
fixture_qrels = {"q1": {"d1": 1}, "q2": {"d2": 1}}
assert validate_retrieval_records(fixture_corpus, fixture_queries, fixture_qrels)
print("normalized retrieval schema test: PASS")

normalized retrieval schema test: PASS


#### D — Why the replacement is correct

A retriever searches the corpus; relevance labels are used for training/evaluation and must point to real corpus IDs. Keeping the three collections independent makes referential integrity testable and prevents answers from being mistaken for retrieved passages.

#### E — Repository change

Refactor `src/data_utils/corpus.py` so it loads the document collection separately from query splits and qrels. Store `self.corpus_records`, `self.query_records` (or split-specific query datasets), and `self.qrels`. Remove `__assign_ids` logic that treats each element of `ground_truths` or each character of `context` as a separate relevant document. Make `all_documents` return the actual corpus collection, not concatenated answer snippets.

### 6.2 FiQA: recommended retrieval source and label semantics

The supplied code uses `load_dataset("vibrantlabsai/fiqa", "main")`. According to that dataset's current Hub card:

- `main` has question text and `ground_truths` answer strings.
- `corpus` is a separate configuration containing the 57,638-document financial corpus.
- The `main` config itself does not contain explicit query-document qrels.

A BEIR-compatible FiQA release has the retrieval structure needed for this project: corpus records, query records and qrels. The Hugging Face cards for `BeIR/fiqa` and `BeIR/fiqa-qrels` describe these structures. Use the BEIR qrels (train/dev/test) for supervised retrieval and evaluation; do not use the reference answer text as a substitute for document IDs.

Example of the expected data shape (the exact loading adapter can be implemented using the BEIR files or the corresponding Hugging Face datasets):

```python
corpus = {
    "doc_id_1": {"title": "", "text": "actual corpus passage ..."},
    "doc_id_2": {"title": "", "text": "another corpus passage ..."},
}
queries = {"query_id_1": "financial question ..."}
qrels = {"query_id_1": {"doc_id_2": 1}}
```

**Important split policy:** train DPR using train qrels only; use dev/validation qrels for model selection; evaluate the final chosen checkpoint once on test qrels. All test queries can retrieve from the same fixed corpus, but test qrels must not be used for training or early stopping.

### 6.3 SQuAD adaptation: deduplicate context paragraphs, never iterate over characters

#### A — Current implementation

The current code loops over `documents = row['text']` after renaming SQuAD's `context` string to `text`. Python string iteration yields one character at a time.

#### B — Why it is wrong

The resulting document IDs are assigned to characters, not passages. Also, many SQuAD rows share the same context paragraph, so a retrieval adaptation should deduplicate identical contexts and let multiple questions point to the same passage ID.

#### C — Proposed implementation

This adapter builds a stable ID from the context text, preserves each context as one complete passage, and creates one positive judgement from each question to its source context. It is a transparent SQuAD-to-retrieval conversion, not an original BEIR-style qrels release.

In [ ]:
import hashlib

def adapt_squad_rows(split_to_rows):
    """Convert SQuAD-style rows into corpus/query/qrels records.

    split_to_rows: {"train": iterable[dict], "validation": iterable[dict], ...}
    Each row contains at least id, question and context.
    """
    corpus_by_id = {}
    query_records = []
    qrels = {}
    query_splits = {}

    for split_name, rows in split_to_rows.items():
        for row in rows:
            context = row["context"].strip()
            question = row["question"].strip()
            if not context or not question:
                continue
            document_id = "squad_" + hashlib.sha256(context.encode("utf-8")).hexdigest()[:20]
            corpus_by_id.setdefault(document_id, {"document_id": document_id, "text": context})
            query_id = f"{split_name}:{row['id']}"
            query_records.append({"query_id": query_id, "text": question})
            qrels[query_id] = {document_id: 1}
            query_splits[query_id] = split_name

    corpus_records = list(corpus_by_id.values())
    validate_retrieval_records(corpus_records, query_records, qrels)
    return corpus_records, query_records, qrels, query_splits

squad_fixture = {
    "train": [
        {"id": "a", "question": "Who built the house?", "context": "The house was built by Ada."},
        {"id": "b", "question": "What was built?", "context": "The house was built by Ada."},
    ],
    "validation": [
        {"id": "c", "question": "Who built it?", "context": "A second paragraph about Grace."},
    ],
}
squad_corpus, squad_queries, squad_qrels, squad_splits = adapt_squad_rows(squad_fixture)
assert len(squad_corpus) == 2                  # repeated context is deduplicated
assert len(squad_queries) == 3
assert squad_qrels["train:a"] == squad_qrels["train:b"]
assert all(len(row["text"]) > 10 for row in squad_corpus)  # not one-character docs
print("SQuAD adapter fixture test: PASS")
print("unique passages:", len(squad_corpus), "queries:", len(squad_queries))

SQuAD adapter fixture test: PASS
unique passages: 2 queries: 3


#### D — Why the replacement is correct

Each passage is a complete string, IDs are deterministic across runs, repeated contexts share one ID, and each question is linked to the paragraph from which its answer was extracted. The adapter can be extended to preserve multiple positive passages when a task supplies them.

#### E — Repository change

Replace the current generic `__assign_ids` logic for SQuAD with an adapter like the above. Keep train/validation query split membership in a separate field. Use the union of unique contexts as the fixed search corpus, and use each split's queries/qrels only for the corresponding training or evaluation phase. State in the report that SQuAD qrels are constructed from source contexts.

### 6.4 `DocumentDataset` zips strings and accidentally creates character documents

#### A — Current implementation

`src/data_utils/dataset.py`:

```python
return [Document(text=text, _id=_id)
        for entries in documents
        for (text, _id) in zip(entries['text'], entries['document_id'])]
```

#### B — Why it is wrong

For normalized records where `text` is a string and `document_id` is a string, `zip` iterates over the two strings character-by-character. The implementation only behaves as intended when both fields are nested lists, which is an undocumented and fragile data contract.

#### C — Proposed implementation

Use one record per document. Accept a Hugging Face Dataset, list of row dictionaries, or other iterable of mappings where every row has a string `text` and a scalar document ID.

In [ ]:
from collections.abc import Mapping
from src.data_utils.document import Document

class DocumentDatasetFixed:
    def __init__(self, documents):
        self.documents = self._build_documents(documents)

    @staticmethod
    def _build_documents(documents):
        output = []
        seen = set()
        for row in documents:
            if not isinstance(row, Mapping):
                raise TypeError("each document row must be a mapping")
            text = row["text"]
            document_id = row.get("document_id", row.get("_id"))
            if not isinstance(text, str):
                raise TypeError("one corpus row must contain one string-valued text field")
            if not isinstance(document_id, (str, int)):
                raise TypeError("document_id must be a scalar string or integer")
            document_id = str(document_id)
            if document_id in seen:
                raise ValueError(f"duplicate document_id: {document_id}")
            if not text.strip():
                raise ValueError(f"empty document text for {document_id}")
            output.append(Document(text=text, _id=document_id))
            seen.add(document_id)
        return output

    def GetDocuments(self):
        return self.documents

# Reproduce the current bug using the original class.
from src.data_utils.dataset import DocumentDataset as DocumentDatasetCurrent
current_result = DocumentDatasetCurrent([{"text": "cat", "document_id": "d1"}]).GetDocuments()
print("current implementation created:", [(d.GetText(), d.GetId()) for d in current_result])

fixed_result = DocumentDatasetFixed([{"text": "cat", "document_id": "d1"}]).GetDocuments()
assert len(fixed_result) == 1
assert fixed_result[0].GetText() == "cat"
assert fixed_result[0].GetId() == "d1"
print("fixed DocumentDataset test: PASS")

current implementation created: [('c', 'd'), ('a', '1')]
fixed DocumentDataset test: PASS


#### D — Why the replacement is correct

It creates exactly one `Document` per corpus row, preserves the full passage text and ID, and raises clear errors for malformed or duplicate records. The normalized corpus contract now matches what TF-IDF, BM25 and dense retrieval expect.

#### E — Repository change

Replace `DocumentDataset.__BuildDocuments` in `src/data_utils/dataset.py` with a one-row-per-document implementation. Update any callers still passing nested arrays so they first flatten to normalized `{text, document_id}` rows.

### 6.5 Offline flags prevent a fresh dataset download

#### A — Current implementation

`src/data_utils/download_retrieval_datasets.py` sets these unconditionally before importing `datasets`:

```python
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
```

#### B — Why it is wrong

A clean environment is forced into offline mode, so `load_dataset(...)` cannot fetch the data unless it is already cached. That conflicts with the apparent intent of the download helpers.

#### C — Proposed implementation

Do not force offline mode in the module. Let users choose it through environment variables or a config value. For example:

```python
# Suggested download_retrieval_datasets.py pattern
from datasets import load_dataset

def get_fiqa_questions(base_dir: str):
    return load_dataset("vibrantlabsai/fiqa", "main", cache_dir=base_dir)

def get_fiqa_corpus(base_dir: str):
    return load_dataset("vibrantlabsai/fiqa", "corpus", cache_dir=base_dir)

def get_squad_dataset(base_dir: str):
    return load_dataset("rajpurkar/squad", cache_dir=base_dir)
```

Choose online/offline mode **before starting Python**, so the Hugging Face Hub and Datasets libraries see the setting during initialization:

```bash
# Normal online run
python main.py

# Explicit offline run; all required files must already be cached
HF_HUB_OFFLINE=1 HF_DATASETS_OFFLINE=1 python main.py
```

PowerShell equivalent:

```powershell
$env:HF_HUB_OFFLINE="1"
$env:HF_DATASETS_OFFLINE="1"
python main.py
```


#### D — Why the replacement is correct

Offline behavior becomes explicit at the call site rather than silently imposed on every user. The example separates FiQA question/answer data from the independent corpus configuration. For a full IR benchmark, use a source with qrels (for example BEIR FiQA), not only these QA rows.

#### E — Repository change

Remove the unconditional `HF_HUB_OFFLINE` and `HF_DATASETS_OFFLINE` assignments from `src/data_utils/download_retrieval_datasets.py`. Add an `offline` option to configuration or document the two environment variables as an explicit user setting.

## 7. Tokenization: a missing contract between text and token IDs

### 7.1 Why the current DPR system cannot reliably consume real text

#### A — Current implementation

The `TransformerEncoder` accepts integer token IDs. The synthetic demo builds a custom word-to-ID dictionary and adds a CLS marker by hand:

```python
ids = [CLS] + [vocab[w] for w in text.split()]
```

There is no tokenizer owned by the repository DPR model, no vocabulary persisted alongside checkpoints, and no clear shared tokenization contract for queries and documents.

#### B — Why it is wrong

The encoder cannot accept raw strings. If query and document token IDs are generated by different vocabularies, or if a vocabulary changes between training and inference, the same integer may represent different words. Unknown tokens, padding, truncation and special tokens must be handled consistently.

A random embedding table plus a custom Transformer also learns language representations from scratch. A small retrieval dataset may not provide enough examples for strong semantic representations. This is a performance risk, not proof of an encoder math defect.

#### C — Proposed implementation

For a minimal scratch-trained baseline, use one deterministic tokenizer for both queries and passages, reserve PAD/UNK/CLS IDs, and save its vocabulary with the model checkpoint. This whitespace tokenizer is deliberately a simple integration smoke test; for better language coverage, use a subword tokenizer. A pretrained tokenizer alone does **not** make a scratch Transformer pretrained—the encoder weights must also be loaded to claim pretrained representations.

In [ ]:
import re

class SimpleVocabularyTokenizer:
    PAD, UNK, CLS = "[PAD]", "[UNK]", "[CLS]"

    def __init__(self):
        self.token_to_id = {self.PAD: 0, self.UNK: 1, self.CLS: 2}

    @staticmethod
    def split(text):
        return re.findall(r"[a-z0-9]+(?:['’][a-z0-9]+)?", text.lower())

    def fit(self, texts):
        for text in texts:
            for token in self.split(text):
                if token not in self.token_to_id:
                    self.token_to_id[token] = len(self.token_to_id)
        return self

    @property
    def vocab_size(self):
        return len(self.token_to_id)

    def encode_batch(self, texts, max_len=64):
        if max_len < 2:
            raise ValueError("max_len must leave room for CLS and at least one content token")
        encoded = []
        for text in texts:
            ids = [self.token_to_id[self.CLS]]
            ids.extend(self.token_to_id.get(t, self.token_to_id[self.UNK]) for t in self.split(text))
            encoded.append(ids[:max_len])
        width = max_len
        batch = torch.full((len(encoded), width), self.token_to_id[self.PAD], dtype=torch.long)
        for i, ids in enumerate(encoded):
            batch[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
        return batch

tok = SimpleVocabularyTokenizer().fit([
    "credit card interest rates", "how does compound interest work", "bank loan rates"
])
example_ids = tok.encode_batch(["How does compound interest work?", "unknown phrase"], max_len=8)
assert example_ids.dtype == torch.long
assert example_ids.shape == (2, 8)
assert int(example_ids.max()) < tok.vocab_size
assert (example_ids[:, 0] == tok.token_to_id[tok.CLS]).all()
print("tokenizer contract test: PASS; vocabulary size:", tok.vocab_size)
print(example_ids)

tokenizer contract test: PASS; vocabulary size: 13
tensor([[ 2,  7,  8,  9,  5, 10,  0,  0],
        [ 2,  1,  1,  0,  0,  0,  0,  0]])


#### D — Why the replacement is correct

One tokenizer owns one mapping; both text types use it; PAD is always zero; unknown words map to UNK; CLS is consistently inserted at position zero; and output tensors are `torch.long` with a predictable fixed length. This closes the interface gap required by `TransformerEncoder`.

#### E — Repository change

Create `src/data_utils/tokenizer.py` (or an equivalent module). Pass the same tokenizer to both towers and the vector database. Persist the token-to-ID mapping or tokenizer files with the model checkpoint. Add tests for vocabulary stability, unknown tokens, padding, truncation and CLS placement.

**Model-quality recommendation:** compare (1) the scratch Transformer with the agreed tokenizer and (2) a pretrained encoder/tokenizer baseline such as BERT or ModernBERT. Keep the custom encoder experiment to satisfy the implementation learning objectives, but do not assume it will outperform a pretrained model without sufficient training data.

## 8. TF-IDF and BM25 baselines: fixes needed before comparing DPR

### 8.1 TF-IDF: three concrete failures and one assignment mismatch

#### A — Current implementation

`src/retrievers/tf_idf.py` contains:

```python
processed_texts = map(...)
all_docs_text = list(*processed_texts)
...
tf = doc_term_count[term] / sum(doc_term_count.values())
...
if term in self.term_to_index():
```

#### B — Why it is wrong

1. `list(*processed_texts)` passes each yielded set as a separate positional argument to `list`. With more than one document it raises `TypeError`; with one document it does not produce a list of document token sets.
2. `doc_term_count[term]` raises `KeyError` whenever a vocabulary term is absent from the current document.
3. `self.term_to_index()` calls a dictionary as a function and raises `TypeError`.
4. The assignment brief defines TF as the term count divided by the maximum term count in that document, while the implementation divides by total document token count. These are different TF conventions. Match the assignment formula if that is what the group intends to report.

#### C — Proposed implementation

Use a materialized list of per-document token sets for document frequency; use `.get(term, 0)` for absent terms; use the dictionary directly; and implement the assignment's max-frequency normalization.

In [ ]:
# Reproduce the current TF-IDF implementation's three direct errors.
from types import SimpleNamespace
from src.retrievers.tf_idf import TFIDF as CurrentTFIDF
from src.data_utils.document import Document

probe = CurrentTFIDF.__new__(CurrentTFIDF)
probe.index = SimpleNamespace(GetDocuments=lambda: [
    Document("apple fruit", "d1"), Document("banana fruit", "d2")
])
probe.corpus_vocabulary = {"apple", "banana", "fruit"}
try:
    probe.GetDocumentFrequencies()
except TypeError as exc:
    print("current GetDocumentFrequencies list(*generator) failure reproduced:", str(exc).splitlines()[0])
else:
    raise AssertionError("Expected current document-frequency code to fail")

probe.idf = {"apple": 1.0, "banana": 1.0, "fruit": 1.0}
try:
    probe.GetDocumentsTFIDFVectors()
except KeyError as exc:
    print("current absent-term KeyError reproduced:", repr(exc))
else:
    raise AssertionError("Expected current TF-IDF matrix builder to fail on absent term")

probe.term_to_index = {"apple": 0, "banana": 1, "fruit": 2}
try:
    probe.QueryToVector(["apple"])
except TypeError as exc:
    print("current dictionary-call TypeError reproduced:", str(exc).splitlines()[0])
else:
    raise AssertionError("Expected current QueryToVector to call a dict as a function")


current GetDocumentFrequencies list(*generator) failure reproduced: list expected at most 1 argument, got 2
current absent-term KeyError reproduced: KeyError('banana')
current dictionary-call TypeError reproduced: 'dict' object is not callable


In [ ]:
from collections import Counter
from math import log
from scipy.sparse import lil_matrix
import re

def preprocess_text(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+(?:['’][a-z0-9]+)?", text.lower())

def build_tfidf_matrix(document_texts):
    tokenized = [preprocess_text(text) for text in document_texts]
    token_sets = [set(tokens) for tokens in tokenized]  # one set per document
    n_docs = len(tokenized)
    if n_docs == 0:
        raise ValueError("TF-IDF requires at least one document")

    vocabulary = sorted(set().union(*token_sets))
    term_to_index = {term: i for i, term in enumerate(vocabulary)}
    document_frequency = {
        term: sum(term in doc_terms for doc_terms in token_sets)
        for term in vocabulary
    }
    idf = {term: log(n_docs / df) for term, df in document_frequency.items()}

    matrix = lil_matrix((n_docs, len(vocabulary)), dtype=np.float32)
    for doc_idx, tokens in enumerate(tokenized):
        counts = Counter(tokens)
        max_tf = max(counts.values(), default=0)
        if max_tf == 0:
            continue
        for term, count in counts.items():
            col = term_to_index[term]
            tf = count / max_tf  # matches assignment's max-frequency TF definition
            matrix[doc_idx, col] = tf * idf[term]

    return matrix.tocsr(), vocabulary, term_to_index, idf, document_frequency

def build_query_matrix(queries, term_to_index):
    query_matrix = lil_matrix((len(queries), len(term_to_index)), dtype=np.float32)
    for row_idx, query in enumerate(queries):
        for term, count in Counter(preprocess_text(query)).items():
            col = term_to_index.get(term)  # None for out-of-vocabulary query terms
            if col is not None:
                query_matrix[row_idx, col] = count
    return query_matrix.tocsr()

#### D — Why the replacement is correct

- Each element of `token_sets` represents exactly one document.
- Document frequency counts each term at most once per document.
- Missing terms contribute zero rather than raising.
- Out-of-vocabulary query terms are ignored safely.
- TF is max-normalized as defined in the supplied assignment. If the project intentionally chooses length-normalized TF instead, document that as a deliberate variant and compare it consistently.

#### E — Repository change

Fix `GetDocumentFrequencies`, `GetDocumentsTFIDFVectors` and `QueryToVector` in `src/retrievers/tf_idf.py`. Add tests with at least two documents where each contains a term missing from the other, plus a query containing an unknown term. Ensure the TF convention matches the formula in the assignment report.

In [ ]:
# Validate TF-IDF on a tiny corpus with disjoint and shared terms.
tfidf_matrix, vocab, term_to_index, idf, df = build_tfidf_matrix([
    "apple apple fruit",
    "banana fruit",
])
q_matrix = build_query_matrix(["apple unknown", "banana"], term_to_index)
scores = (q_matrix @ tfidf_matrix.T).toarray()
assert tfidf_matrix.shape == (2, len(vocab))
assert q_matrix.shape == (2, len(vocab))
assert scores.shape == (2, 2)
assert np.isfinite(scores).all()
assert scores[0, 0] > scores[0, 1]
assert scores[1, 1] > scores[1, 0]
print("TF-IDF matrix/query tests: PASS")
print("vocabulary:", vocab)
print("scores:\n", scores.round(4))

TF-IDF matrix/query tests: PASS
vocabulary: ['apple', 'banana', 'fruit']
scores:
 [[0.69 0.  ]
 [0.   0.69]]


### 8.2 BM25: matrix is never initialized and absent terms crash

#### A — Current implementation

`src/retrievers/bm25.py` defines `GetDocumentsBM25Vectors`, but the constructor only assigns `self.idf_bm25`. `CalculateScores` later reads `self.bm25_matrix`. The matrix builder also uses `doc_term_count[term]` without a default.

#### B — Why it is wrong

The inherited `TFIDF.__init__` first calls the shared document-frequency method, which has the TF-IDF bug documented above; after that shared bug is fixed, `self.bm25_matrix` still does not exist when `CalculateScores` is called, so scoring fails. Even if it were built manually, absent terms trigger `KeyError`. The average document length also needs protection for an empty or all-empty corpus.

#### C — Proposed implementation

Initialize the matrix during construction and use zero counts for missing terms. Calculate document lengths once and reject a zero average length.

In [ ]:
# Reproduce the current BM25 CalculateScores access to a matrix never initialized.
from scipy.sparse import lil_matrix
from src.retrievers.bm25 import BM25 as CurrentBM25
bm25_probe = CurrentBM25.__new__(CurrentBM25)
bm25_probe.QueryToVector = lambda queries: lil_matrix((len(queries), 1), dtype=np.float32).tocsr()
try:
    bm25_probe.CalculateScores(["apple"])
except AttributeError as exc:
    print("current BM25 missing bm25_matrix failure reproduced:", str(exc).splitlines()[0])
else:
    raise AssertionError("Expected current BM25 score method to access missing bm25_matrix")


current BM25 missing bm25_matrix failure reproduced: 'BM25' object has no attribute 'bm25_matrix'


In [ ]:
# Suggested BM25 initialization and matrix builder inside BM25.
# After super().__init__(documents, index_path):
#     self.k1 = k1
#     self.b = b
#     self.idf_bm25 = self.GetInverseDocumentFrequenciesBM25()
#     self.bm25_matrix = self.GetDocumentsBM25Vectors()

# Replacement pattern for GetDocumentsBM25Vectors:
def build_bm25_matrix(document_texts, vocabulary, idf_bm25, k1=1.5, b=0.75):
    tokenized = [preprocess_text(text) for text in document_texts]
    lengths = [len(tokens) for tokens in tokenized]
    if not lengths:
        raise ValueError("BM25 requires at least one document")
    avg_len = float(np.mean(lengths))
    if avg_len <= 0:
        raise ValueError("BM25 requires a corpus with positive average document length")

    term_to_index = {term: i for i, term in enumerate(vocabulary)}
    matrix = lil_matrix((len(tokenized), len(vocabulary)), dtype=np.float32)
    for doc_idx, tokens in enumerate(tokenized):
        counts = Counter(tokens)
        doc_len = lengths[doc_idx]
        for term in vocabulary:
            tf = counts.get(term, 0)
            if tf == 0:
                continue
            denominator = tf + k1 * (1 - b + b * doc_len / avg_len)
            saturated_tf = tf * (k1 + 1) / denominator
            matrix[doc_idx, term_to_index[term]] = saturated_tf * idf_bm25[term]
    return matrix.tocsr(), term_to_index

#### D — Why the replacement is correct

It creates the attribute that `CalculateScores` expects, retains BM25's term-frequency saturation and document-length normalization, and treats terms absent from a passage as zero. It also avoids an undefined average length.

#### E — Repository change

In `src/retrievers/bm25.py`, assign `self.bm25_matrix = self.GetDocumentsBM25Vectors()` in the constructor after `k1`, `b` and `idf_bm25` are initialized. Use `.get(term, 0)` in the matrix builder, compute document lengths once, and add tests that construct and score BM25 on a two-document corpus with non-overlapping terms.

In [ ]:
# Validate the proposed BM25 matrix and ranking.
docs_for_bm25 = ["apple apple fruit", "banana fruit"]
tok_docs = [preprocess_text(x) for x in docs_for_bm25]
all_terms = sorted(set().union(*(set(x) for x in tok_docs)))
n_docs = len(tok_docs)
df_bm25 = {t: sum(t in set(d) for d in tok_docs) for t in all_terms}
idf_bm25 = {t: np.log((n_docs + 1) / (df_bm25[t] + 0.5)) for t in all_terms}
bm25_matrix, bm25_term_index = build_bm25_matrix(docs_for_bm25, all_terms, idf_bm25)
bm25_q = build_query_matrix(["apple unknown", "banana"], bm25_term_index)
bm25_scores = (bm25_q @ bm25_matrix.T).toarray()
assert bm25_scores.shape == (2, 2) and np.isfinite(bm25_scores).all()
assert bm25_scores[0, 0] > bm25_scores[0, 1]
assert bm25_scores[1, 1] > bm25_scores[1, 0]
print("BM25 matrix/query tests: PASS")
print(bm25_scores.round(4))

BM25 matrix/query tests: PASS
[[0.93 0.  ]
 [0.   0.76]]


## 9. Evaluation: correct Recall@k and add Precision@k / MRR

### 9.1 Current recall denominator can overstate recall

#### A — Current implementation

`src/evaluation/metrics.py`:

```python
def calculateRecall(relevancies, query):
    return sum(relevancies) / min(len(relevancies), query.getNumberOfRelevantDocuments())
```

#### B — Why it is wrong

Recall@k is:

\[
\operatorname{Recall@k}(q)=\frac{|\text{relevant retrieved in top-k}|}{|\text{all relevant documents for }q|}.
\]

The denominator is the total number of known relevant documents, not `min(k, number_of_relevant_documents)`. For example, if a query has 4 relevant documents and the top-2 contains 2 of them, recall is 2/4 = 0.5. The current implementation divides by `min(2,4)=2` and reports 1.0.

The current evaluator also does not implement Precision@k or MRR, both required by the assignment.

#### C — Proposed implementation

Evaluate ranked document IDs directly against qrels. This makes the metric definitions explicit and avoids depending on mutable `Query` object methods.

In [ ]:
# Reproduce the existing Recall@k denominator error.
from src.evaluation.metrics import calculateRecall as current_calculate_recall
from src.data_utils.query import Query as CurrentQuery
q = CurrentQuery("example", id="q1", relevant_document_ids=["d1", "d2", "d3", "d4"])
current_value = current_calculate_recall([True, True], q)
assert current_value == 1.0  # current code divides by min(k, number of relevant docs)
assert current_value != 0.5  # correct Recall@2 is 2/4
print("current Recall@2 overstatement reproduced: reports", current_value, "instead of 0.5")


current Recall@2 overstatement reproduced: reports 1.0 instead of 0.5


In [ ]:
def precision_at_k(ranked_ids, relevant_ids, k):
    if k <= 0:
        raise ValueError("k must be positive")
    relevant = set(relevant_ids)
    return sum(doc_id in relevant for doc_id in ranked_ids[:k]) / k

def recall_at_k(ranked_ids, relevant_ids, k):
    if k <= 0:
        raise ValueError("k must be positive")
    relevant = set(relevant_ids)
    if not relevant:
        raise ValueError("recall is undefined for a query with no relevant documents")
    return sum(doc_id in relevant for doc_id in ranked_ids[:k]) / len(relevant)

def reciprocal_rank(ranked_ids, relevant_ids):
    relevant = set(relevant_ids)
    for rank, doc_id in enumerate(ranked_ids, start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

def evaluate_rankings(ranked_by_query, qrels, k=10):
    query_ids = [qid for qid in ranked_by_query if qid in qrels and qrels[qid]]
    if not query_ids:
        raise ValueError("No evaluable queries with positive qrels")
    p_values, r_values, rr_values = [], [], []
    for qid in query_ids:
        ranked = ranked_by_query[qid]
        relevant = {doc_id for doc_id, score in qrels[qid].items() if score > 0}
        p_values.append(precision_at_k(ranked, relevant, k))
        r_values.append(recall_at_k(ranked, relevant, k))
        rr_values.append(reciprocal_rank(ranked, relevant))
    return {
        f"precision@{k}": float(np.mean(p_values)),
        f"recall@{k}": float(np.mean(r_values)),
        "mrr": float(np.mean(rr_values)),
        "num_queries": len(query_ids),
    }

#### D — Why the replacement is correct

Precision divides by the requested cutoff `k`; recall divides by the total number of qrel-positive documents for that query; reciprocal rank is zero when no relevant document is retrieved and otherwise the reciprocal of the first relevant rank. The evaluator macro-averages per-query values over queries with positive qrels.

The convention for Precision@k when fewer than `k` results are returned is denominator `k`, matching the common fixed-cutoff definition. The index should normally return `k` results whenever the corpus has at least `k` documents.

#### E — Repository change

Replace `src/evaluation/metrics.py` with these metrics or equivalent functions. Update `main.py` / experiment runner to pass ranked document IDs and qrels. Report the chosen metric conventions in the project report.

In [ ]:
# Metric validation: q1 has 4 relevant documents; top-2 retrieves 2 => recall 0.5, not 1.0.
ranked = {"q1": ["d1", "x", "d2"], "q2": ["z", "d3"]}
qrels_test = {"q1": {"d1": 1, "d2": 1, "d4": 1, "d5": 1},
              "q2": {"d3": 1}}
metric_values = evaluate_rankings(ranked, qrels_test, k=2)
assert abs(metric_values["precision@2"] - 0.5) < 1e-9
assert abs(metric_values["recall@2"] - 0.625) < 1e-9  # mean of 0.25 and 1.0
assert abs(metric_values["mrr"] - 0.75) < 1e-9       # mean of 1.0 and 0.5
print(metric_values)
print("Precision@k / Recall@k / MRR tests: PASS")

{'precision@2': 0.5, 'recall@2': 0.625, 'mrr': 0.75, 'num_queries': 2}
Precision@k / Recall@k / MRR tests: PASS


## 10. Main program and Hydra configuration do not select DPR

### 10.1 The main program hard-codes TF-IDF

#### A — Current implementation

`main.py` imports `TFIDF` and does:

```python
tf_idf = TFIDF(documents=corpus.all_documents)
...
```

It never reads `cfg.retriever.method`. The Hydra config declares a retriever group, but the selected method is not used by `main.py`.

#### B — Why it is wrong

Changing the configuration from `tf_idf` to `bm25` or `dpr` does not switch the executed retrieval implementation. The model config is also not composed in the experiment defaults. It is easy to believe an experiment ran DPR when it actually ran TF-IDF.

#### C — Proposed implementation

Make the experiment runner dispatch explicitly on the configured retriever method. The following is the intended structure; the DPR branch should only be enabled after the DPR and data/training work above is complete.

```python
# Suggested dispatch structure for main.py (illustrative until retriever interfaces are unified).
method = str(cfg.retriever.method).lower().replace("-", "_")

if method in {"tf_idf", "tfidf", "td_idf"}:
    retriever = TFIDF(documents=corpus.all_documents)
elif method == "bm25":
    retriever = BM25(documents=corpus.all_documents)
elif method == "dpr":
    # Load tokenizer + trained DPR checkpoint + document index here.
    # Do not silently train or initialize random encoders in the evaluation branch.
    retriever = load_trained_dpr_retriever(cfg, corpus)
else:
    raise ValueError(f"Unsupported retriever method: {cfg.retriever.method}")
```

#### D — Why the replacement is correct

The configured experiment and the actual model now agree. The DPR evaluation path requires a trained checkpoint and an index built using the matching document encoder/tokenizer. It must not silently fall back to random weights or another retriever.

#### E — Repository change

1. Import `BM25` and the DPR retrieval adapter in `main.py`.
2. Branch on `cfg.retriever.method`.
3. Add `- model: encoder` to `config/fiqa_experiment.yaml` if the `model` group is used.
4. Keep retriever training separate from final evaluation: first train/save, then build index, then evaluate the selected checkpoint.
5. Add a test or log line that prints the concrete retriever class and checkpoint path for each run.

## 11. Vector database and checkpoint consistency

### 11.1 Safe encoding, index arguments and embedding provenance

#### A — Current implementation

In `src/retrievers/vector_database.py`, the current encoder helper follows this pattern:

```python
@torch.no_grad()
def encode_token_ids(encoder, token_ids, batch_size=64):
    was_training = encoder.training
    encoder.eval()
    out = []
    for i in range(0, len(token_ids), batch_size):
        out.append(encoder(token_ids[i:i + batch_size]))
    encoder.train(was_training)
    return torch.cat(out)
```

The current `FlatIndex.search` also passes `k` into NumPy's partitioning logic without validating that `k` is a positive integer. The persisted database metadata records index information, but not a strong identity for the exact encoder/tokenizer state that created its vectors.

#### B — Why it is wrong / risky

- An empty input produces no output batches, so `torch.cat([])` raises an exception.
- Token IDs are not explicitly moved to the encoder's device, which can create CPU/GPU device errors.
- If encoding raises, the encoder may remain in evaluation mode because restoration is not in `finally`.
- A nonpositive `k` is not a meaningful top-k request; depending on the value and corpus size, the current code can return an empty or nonsensical result instead of raising a clear validation error.
- A vector index built with one document encoder or vocabulary is not generally compatible with another. A model update without rebuilding the index can silently invalidate retrieval.

#### C — Proposed implementation

Use the safe encoding helper in the next code cell. Add a positive-integer check to `FlatIndex.search` and `VectorDatabase.query`, and record checkpoint/tokenizer identity, pooling, and embedding dimension in index metadata.


In [ ]:
@torch.no_grad()
def encode_token_ids_safe(encoder, token_ids, batch_size=64):
    if not isinstance(token_ids, torch.Tensor) or token_ids.ndim != 2:
        raise ValueError("token_ids must be a rank-2 torch.Tensor with shape (N, L)")
    if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size <= 0:
        raise ValueError("batch_size must be a positive integer")
    try:
        device = next(encoder.parameters()).device
    except StopIteration:
        device = token_ids.device

    was_training = encoder.training
    outputs = []
    try:
        encoder.eval()
        for start in range(0, len(token_ids), batch_size):
            batch = token_ids[start:start + batch_size].to(device)
            outputs.append(encoder(batch))
    finally:
        encoder.train(was_training)

    if not outputs:
        output_dim = getattr(encoder, "d_model", None)
        if output_dim is None:
            raise ValueError("Cannot infer embedding dimension for an empty input batch")
        return torch.empty((0, output_dim), device=device)
    return torch.cat(outputs, dim=0)


def validate_top_k(k):
    if not isinstance(k, int) or isinstance(k, bool) or k <= 0:
        raise ValueError("k must be a positive integer")
    return k


In [ ]:
# Check nonempty, empty and exceptional encoding paths, plus invalid top-k arguments.
small_encoder = TransformerEncoder(
    vocab_size=20, d_model=8, n_heads=2, n_layers=1,
    feed_forward_dim=16, max_len=8, dropout=0.0, pooling_type="mean"
)
small_encoder.train()
ids = torch.tensor([[2, 3, 0], [2, 4, 5]], dtype=torch.long)
encoded = encode_token_ids_safe(small_encoder, ids, batch_size=1)
assert encoded.shape == (2, 8) and torch.isfinite(encoded).all()
assert small_encoder.training is True
empty = encode_token_ids_safe(small_encoder, torch.empty((0, 3), dtype=torch.long))
assert empty.shape == (0, 8)
for bad_k in [0, -1, 1.5, True]:
    try:
        validate_top_k(bad_k)
    except ValueError:
        pass
    else:
        raise AssertionError(f"invalid k accepted: {bad_k!r}")
assert validate_top_k(5) == 5
print("vector encoding, empty batch, mode restoration and top-k validation: PASS")


class FailingEncoder(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = torch.nn.Parameter(torch.ones(1))
        self.d_model = 1
    def forward(self, token_ids):
        raise RuntimeError("deliberate test failure")

failing_encoder = FailingEncoder().train()
try:
    encode_token_ids_safe(failing_encoder, torch.ones((1, 2), dtype=torch.long))
except RuntimeError as exc:
    assert str(exc) == "deliberate test failure"
else:
    raise AssertionError("expected the deliberate encoder exception")
assert failing_encoder.training is True
print("encoder mode restored after exception: PASS")


vector encoding, empty batch, mode restoration and top-k validation: PASS
encoder mode restored after exception: PASS


### D — Why the replacement is correct

It preserves the encoder's top-level training/evaluation state even if encoding raises, uses the model's device, handles empty batches explicitly and rejects invalid batch sizes/top-k values. Index metadata also gives the retrieval pipeline a way to detect that stored passage vectors belong to a different encoder/tokenizer version.

### E — Repository change

Update `encode_token_ids` in `src/retrievers/vector_database.py` with the validated behavior. Validate `k` in `FlatIndex.search` and `VectorDatabase.query`. Extend the saved `encoder_info` in `VectorDatabase.save` with a checkpoint identifier/hash, tokenizer identifier or vocabulary hash, pooling configuration and embedding dimension. Rebuild the passage index whenever the document encoder or tokenizer changes.

## 12. Save and reload the model and tokenizer as one artifact

#### A — Current implementation

The current vector database saves passage vectors, IDs and database configuration. The repository does not provide a complete DPR checkpoint format that reliably couples both encoder towers with the tokenizer vocabulary and the encoder configuration.

#### B — Why it is wrong / risky

Dense vectors are meaningful only relative to the model and token-to-ID mapping that produced them. If the model weights, tokenizer vocabulary, pooling method or dimensions change, previously indexed vectors may no longer be comparable to fresh query vectors. Saving only the vector index is therefore not enough to reproduce a DPR system.

#### C — Proposed implementation

Save both encoder state dictionaries, the exact encoder configuration, the tokenizer mapping and a format version. On reload, restore both towers and the token mapping before encoding queries or rebuilding/loading an index.


In [ ]:
import torch
from pathlib import Path

def save_dpr_bundle(model, tokenizer, encoder_config, output_path):
    payload = {
        "format_version": 1,
        "query_encoder_state_dict": model.query_encoder.state_dict(),
        "document_encoder_state_dict": model.document_encoder.state_dict(),
        "encoder_config": dict(encoder_config),
        "token_to_id": dict(tokenizer.token_to_id),
    }
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, output_path)
    return output_path


def load_dpr_bundle(checkpoint_path, model, tokenizer, map_location="cpu"):
    payload = torch.load(checkpoint_path, map_location=map_location, weights_only=True)
    if payload.get("format_version") != 1:
        raise ValueError("Unsupported DPR checkpoint format")
    model.query_encoder.load_state_dict(payload["query_encoder_state_dict"])
    model.document_encoder.load_state_dict(payload["document_encoder_state_dict"])
    token_to_id = payload["token_to_id"]
    if token_to_id.get(tokenizer.PAD) != 0 or tokenizer.UNK not in token_to_id or tokenizer.CLS not in token_to_id:
        raise ValueError("Checkpoint tokenizer is missing required special tokens")
    tokenizer.token_to_id = dict(token_to_id)
    return dict(payload["encoder_config"])

print("DPR checkpoint save/load helpers defined.")


DPR checkpoint save/load helpers defined.


In [ ]:
# Round-trip both towers and tokenizer using a small temporary checkpoint.
from tempfile import TemporaryDirectory

checkpoint_tokenizer = SimpleVocabularyTokenizer().fit(["credit risk", "interest rates"])
checkpoint_config = dict(
    d_model=8, n_heads=2, n_layers=1, feed_forward_dim=16,
    max_len=8, padding_idx=0, dropout=0.0, pooling_type="mean"
)
source_model = DPRModel(checkpoint_tokenizer.vocab_size, checkpoint_config)
restored_model = DPRModel(checkpoint_tokenizer.vocab_size, checkpoint_config)
restored_tokenizer = SimpleVocabularyTokenizer()

with TemporaryDirectory() as temp_dir:
    checkpoint_file = Path(temp_dir) / "dpr_bundle.pt"
    save_dpr_bundle(source_model, checkpoint_tokenizer, checkpoint_config, checkpoint_file)
    restored_config = load_dpr_bundle(checkpoint_file, restored_model, restored_tokenizer)
    assert restored_config == checkpoint_config
    assert restored_tokenizer.token_to_id == checkpoint_tokenizer.token_to_id
    for name, tensor in source_model.state_dict().items():
        assert torch.equal(tensor, restored_model.state_dict()[name]), name

print("DPR model + tokenizer checkpoint round-trip: PASS")


DPR model + tokenizer checkpoint round-trip: PASS


#### D — Why the replacement is correct

The checkpoint preserves both towers, the vocabulary mapping and the configuration needed to reconstruct the encoder shape. The round-trip test verifies that both state dictionaries and tokenizer IDs survive saving and reloading. A project-specific tokenizer may need its own `save_pretrained`/`from_pretrained` methods; the core requirement is that the exact tokenizer state is persisted with the model.

#### E — Repository change

Add checkpoint utilities (for example `src/retrievers/checkpoint.py`) with `save_dpr_bundle` and `load_dpr_bundle`. Call save after selecting the best validation checkpoint in `src/training/trainer.py`. Call load before constructing query embeddings in the DPR retriever. Store the checkpoint hash and tokenizer hash in `VectorDatabase.encoder_info`; invalidate/rebuild the index if either hash changes.

## 13. Dependency file cleanup

#### A — Current implementation

The supplied fork's `requirements.txt` contains:

```text
hydra-core>=1.4.0rc3
transformers
torch
datasets
scipy
re
```

#### B — Why it is wrong / unnecessary

`re` is part of the Python standard library and is not installed from PyPI. Listing it as a project dependency is unnecessary and may cause package-resolution confusion. The code should import the standard-library module without adding it to the dependency list.

#### C — Proposed implementation

Remove only the `re` line. Keep actual third-party packages in the requirements file, and consider pinning tested versions in a lockfile or constrained requirements after validating the team's Python/PyTorch environment. Do not arbitrarily change version pins as part of this bug fix.

#### D — Why the replacement is correct

The standard library ships with Python; no third-party installation is required for `import re`. Removing it reduces unnecessary dependency resolution without changing runtime behavior.

#### E — Repository change

Edit `requirements.txt` and delete the standalone `re` entry. Reinstall in a clean virtual environment and run the full project test suite. Keep `re` imports in source files unchanged.


## 14. Training and model-quality protocol

Once the infrastructure is corrected, use this staged validation protocol. A single high training score is not enough.

1. **Unit tests:** shape, mask, finite values, gradients, tokenizer and checkpoint round-trip.
2. **Overfit a tiny clean set:** the model should reduce training contrastive loss on a tiny deterministic positive-pair dataset. This checks that the forward/loss/optimizer path is connected.
3. **Validate retrieval, not just loss:** encode the corpus, run query retrieval and compute Precision@k, Recall@k and MRR on a validation split.
4. **Compare against baselines:** TF-IDF and BM25 must run on exactly the same corpus, queries, qrels and candidate set.
5. **Use a pretrained baseline:** evaluate a pretrained encoder (e.g. BERT/ModernBERT or a sentence embedding checkpoint) separately from the scratch encoder. The assignment allows pretrained Transformer encoders; report the model provenance clearly.
6. **Monitor false negatives:** in-batch negatives should not be known relevant passages. Use a positive mask or a sampler that checks qrels.
7. **Freeze test use:** do not select hyperparameters based on the test split.
8. **Rebuild the index after every final document-encoder checkpoint change.**

Useful training diagnostics to log per epoch: train loss, validation loss, Recall@10, MRR, mean query/document embedding norm, learning rate, gradient norm, and number of positive/negative candidates per query.

**Interpretation:** if the encoder unit tests pass and the model can overfit a tiny set but retrieval quality on validation remains weak, investigate data quality, training pairs, false negatives, batch construction, pooling and tokenizer before changing attention equations. If tiny-set loss cannot decrease, inspect optimizer wiring, gradient flow, label alignment and model initialization.

## 15. Repository change checklist (the actionable proposal)

This is the concise list to take into the code review / discussion with your colleague.

| Order | File | Required change | Acceptance test |
|---:|---|---|---|
| 1 | `src/retrievers/models/encoder.py` | Reject fully masked attention rows; validate masks; optionally expose per-layer attention maps; preserve existing default API | No NaNs; masked weights zero; attention maps returned with expected shapes |
| 2 | `config/model/encoder.yaml` | Replace stale `emb_dim`/`hidden_dim`; add max length, padding, dropout; make pooling consistent | Model config can instantiate encoder using tokenizer vocab size |
| 3 | `config/fiqa_experiment.yaml` | Compose model config and ensure retriever config is used | Config inspection shows model and retriever groups as expected |
| 4 | `src/data_utils/download_retrieval_datasets.py` | Remove forced HF offline mode; separate FiQA questions from corpus | Fresh download works; offline mode only when requested |
| 5 | `src/data_utils/corpus.py` | Separate corpus, queries and qrels; use genuine document corpus and relevance judgments | All qrel IDs resolve to corpus/query IDs; no answer strings treated as documents |
| 6 | `src/data_utils/dataset.py` | One normalized row becomes one full `Document`; no string zipping | A 1-row corpus with `text='abc'` creates exactly one document `abc` |
| 7 | New `src/data_utils/tokenizer.py` | One tokenizer shared by both towers; PAD/UNK/CLS; persisted vocabulary | Stable IDs, correct padding, unknown-token and max-length tests |
| 8 | `src/retrievers/dpr.py` | Implement `nn.Module` with `self.query_encoder` and `self.document_encoder`; load state dicts independently | Both towers construct, forward and reload checkpoints |
| 9 | `src/training/loss.py` | Implement in-batch contrastive loss; support positive masks for multiple positives | Loss finite; labels align; gradients flow; multi-positive test passes |
| 10 | `src/training/trainer.py` | Implement optimizer loop, validation, clipping and checkpointing | A training step changes weights and loss can fall on tiny fixture |
| 11 | `src/retrievers/tf_idf.py` | Fix document frequency, absent terms, query dictionary access; match assignment TF | Two-document ranking fixture passes |
| 12 | `src/retrievers/bm25.py` | Build matrix in constructor; fix absent terms and empty-length guard | BM25 scores and ranks fixture correctly |
| 13 | `src/evaluation/metrics.py` | Correct Recall@k; add Precision@k and MRR | Hand-calculated metric fixture passes |
| 14 | `src/retrievers/vector_database.py` | Validate k/batch sizes/devices; preserve model mode; attach checkpoint/tokenizer metadata | Empty batch, k edge, save/load and matching checkpoint tests |
| 15 | `main.py` | Dispatch on `cfg.retriever.method`; never silently run TF-IDF for DPR config | Log and test concrete retriever selected |
| 16 | `requirements.txt` | Remove `re` (Python standard-library module, not an install dependency); keep only actual third-party dependencies and local project install | `pip install -r requirements.txt` resolves in a clean environment |
| 17 | `src/retrievers/checkpoint.py` | Save/load both towers, tokenizer mapping and encoder config together | Checkpoint round-trip preserves both encoders and vocabulary |

### Suggested implementation order

**Phase A — trustworthy data and evaluation:** items 4–7, 11–13.  
**Phase B — working DPR path:** items 1–3, 8–10, 14–15.  
**Phase C — quality experiments:** compare scratch vs pretrained; tune learning rate, temperature, batch size, max length, pooling and negatives using validation only.

Do not merge everything in one large commit. Use small commits with the acceptance tests above so regressions can be localized.


## 16. Validation results and limits of this notebook

### Tests performed here

- The supplied test suite: encoder + vector database tests.
- An 8-layer forward/backward finite-value stress test on the existing encoder.
- Reproduction of the `DocumentDataset` character-splitting behavior.
- Proposed safe-attention tests, including an all-masked row.
- Proposed SQuAD paragraph de-duplication and query-to-context qrel test.
- Proposed tokenizer contract test.
- Proposed DPR loss, multi-positive mask and gradient tests.
- Proposed tiny synthetic DPR training smoke test.
- Proposed TF-IDF and BM25 matrix/ranking tests.
- Hand-computed Precision@k, Recall@k and MRR tests.

### What is not validated here

- No real FiQA or SQuAD dataset was downloaded in this notebook run.
- No real FiQA qrels were used to train or evaluate DPR.
- No actual end-to-end training run was performed on the project's data.
- The proposed full repository patches have not been applied to the supplied repository; helper/reference implementations in this notebook are tested in isolation.
- A successful synthetic training smoke test is only evidence that the proposed components connect. It is not evidence of strong semantic retrieval quality.

### Environment-specific notes

The review environment used for preparing this notebook had PyTorch, NumPy, SciPy and pytest available. Hugging Face `datasets`, `transformers` and Hydra were not installed in that environment. Therefore dataset download, Hydra composition and pretrained model loading must be validated in the team's clean project environment after the relevant dependencies are installed.

## 17. External dataset references

These are used only to verify dataset schema and retrieval-label availability; repository code findings come from the supplied ZIP snapshot.

1. **vibrantlabsai/fiqa dataset card** — describes the `main` config as question + ground-truth-answer data and a separate `corpus` config with 57,638 documents:  
   https://huggingface.co/datasets/vibrantlabsai/fiqa
2. **BeIR/fiqa dataset card** — describes the retrieval corpus and query data, and FiQA's retrieval benchmark context:  
   https://huggingface.co/datasets/BeIR/fiqa
3. **BeIR/fiqa-qrels dataset card** — describes qrels as query ID, corpus ID and relevance score; the dataset has train/dev/test splits:  
   https://huggingface.co/datasets/BeIR/fiqa-qrels
4. **rajpurkar/squad dataset card** — describes SQuAD fields, including string-valued `context`, question, and answer spans:  
   https://huggingface.co/datasets/rajpurkar/squad
5. **Assignment PDF** — use the supplied assignment's required TF-IDF/BM25, Transformer, DPR, vector index, Precision@k, Recall@k, MRR and attention analysis objectives as the project acceptance target.

---

## 18. Final recommendation

The most important proposal to take to the team is:

> **Keep the current Transformer encoder as the starting implementation, harden its edge cases and observability, but do not treat it as the main cause of weak retrieval results. The current project-level DPR path is unfinished, and the current data preparation does not produce a valid query/corpus/qrels retrieval problem. Correct those contracts, implement a real tokenizer + dual-encoder + contrastive trainer, repair the baselines and evaluation, and only then judge model quality.**

The project is ready for a controlled refactor, not a blind Transformer rewrite.